# Notebook 05 — Evaluation and Paper Figures

Loads baseline scores from Notebook 04, computes KCS-DT statistics, runs statistical tests, and produces all figures for Paper 2.

**No GPU required.**

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import os, sys, json
REPO = 'ifc-graphrag-dt'

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git

!bash {REPO}/colab_setup.sh

os.chdir(REPO)
REPO = '.'   # paths after chdir must not include repo name
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working directory: {os.getcwd()}')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

from evaluation.metrics.kcs_dt import KCSDTScorer
from evaluation.metrics.ggs import GGSScorer
from evaluation.results.statistical_tests import (
    wilcoxon_signed_rank, cohen_d, bootstrap_ci, run_full_analysis
)

SCORES_DIR  = 'outputs/scores'
FIGURES_DIR = 'outputs/figures'
RESULTS_DIR = 'outputs/results'
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Try to load from Drive if available
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_SCORES = '/content/drive/MyDrive/ifc_graphrag_dt_outputs/scores'
    if os.path.exists(DRIVE_SCORES):
        import shutil
        shutil.copytree(DRIVE_SCORES, SCORES_DIR, dirs_exist_ok=True)
        print(f'Scores loaded from Drive: {SCORES_DIR}')
except Exception:
    print('Drive not available — using local outputs/scores/')

BASELINE_NAMES = {'b0':'B0: No Grounding','b1':'B1: Prompt LLM',
                  'b2':'B2: Flat RAG','b3':'B3: IFC Lookup','b4':'B4: GraphRAG-DT'}
COLORS = {'b0':'#888780','b1':'#B4B2A9','b2':'#5DCAA5','b3':'#EF9F27','b4':'#2E5FA3'}
print('Imports OK')

In [ ]:
# ── Cell 2: Load scores ───────────────────────────────────────────────────────
all_scores = {}
for b in ['b0','b1','b2','b3','b4']:
    p = Path(f'{SCORES_DIR}/{b}_scores.json')
    if p.exists():
        with open(p) as f:
            all_scores[b] = json.load(f)
        print(f'{b.upper()}: {len(all_scores[b])} records loaded')
    else:
        print(f'{b.upper()}: NOT FOUND — run Notebook 04 first')
        all_scores[b] = []

In [ ]:
# ── Cell 3: Compute KCS-DT proxy scores ──────────────────────────────────────
# Proxy: normalise entity/relation counts against tier expectations.
# Replace with expert-annotated Stage B scores after annotation phase.

EXPECTED = {1: (1, 0), 2: (3, 2), 3: (7, 5)}  # (entities, relations) per tier

def kcs_proxy(records):
    out = []
    for r in records:
        if r.get('status','ok') != 'ok': continue
        tier = r.get('tier', 1)
        exp_e, exp_r = EXPECTED[tier]
        e  = min(r.get('entities',0)  / exp_e, 1.0) if exp_e  > 0 else 1.0
        rv = min(r.get('relations',0) / exp_r, 1.0) if exp_r  > 0 else 1.0
        kcs = 0.20*e + 0.35*rv + 0.45  # A+Cn+Cv default 1.0 → 0.45
        out.append({'prompt_id': r['prompt_id'], 'tier': tier,
                    'domain': r.get('domain','MEP'), 'total': round(kcs,4),
                    'entity': e, 'relation': rv, 'baseline': r.get('baseline','')})
    return out

kcs = {b: kcs_proxy(recs) for b, recs in all_scores.items()}

print('KCS-DT proxy scores (replace with annotated scores post-annotation):')
print(f'{"Baseline":<8} {"N":>4} {"Mean":>8} {"Std":>8}')
print('-' * 32)
for b, scores in kcs.items():
    if scores:
        vals = [s['total'] for s in scores]
        print(f'{b.upper():<8} {len(vals):>4} {np.mean(vals):>8.4f} {np.std(vals):>8.4f}')

In [ ]:
# ── Cell 4: Table 1 — KCS-DT by baseline and tier ────────────────────────────
print('TABLE 1: KCS-DT (mean ± std) by Baseline and Tier')
print('='*70)
print(f'{"Baseline":<22} {"Tier 1 (Asset)":>18} {"Tier 2 (Assembly)":>19} {"Tier 3 (System)":>17}')
print('-'*70)

table = {}
for b in ['b0','b1','b2','b3','b4']:
    row = []
    for tier in [1, 2, 3]:
        vs = [s['total'] for s in kcs.get(b,[]) if s['tier']==tier]
        if vs:
            row.append((np.mean(vs), np.std(vs), vs))
            cell = f'{np.mean(vs):.3f} ± {np.std(vs):.3f}'
        else:
            row.append((0,0,[]))
            cell = 'N/A'
        if tier == 1:
            print(f'{BASELINE_NAMES[b]:<22}', end='')
        print(f' {cell:>19}', end='')
    print()
    table[b] = row
print('='*70)

In [ ]:
# ── Cell 5: Figure 1 — KCS-DT grouped bar chart ───────────────────────────────
tiers = [1, 2, 3]
x = np.arange(len(tiers))
w = 0.15
baselines = ['b0','b1','b2','b3','b4']

fig, ax = plt.subplots(figsize=(13, 6))
for i, b in enumerate(baselines):
    means = [table[b][t][0] for t in range(3)]
    stds  = [table[b][t][1] for t in range(3)]
    offset = (i - len(baselines)/2 + 0.5) * w
    ax.bar(x + offset, means, w, label=BASELINE_NAMES[b], color=COLORS[b],
           yerr=stds, capsize=3, edgecolor='white', linewidth=0.5)

ax.axhline(y=0.73, color='#2E5FA3', linestyle='--', alpha=0.4, linewidth=1.5, label='H1 target (0.73)')
ax.set_xticks(x)
ax.set_xticklabels(['Tier 1\n(Asset)', 'Tier 2\n(Assembly)', 'Tier 3\n(System)'], fontsize=11)
ax.set_ylabel('KCS-DT Score', fontsize=12)
ax.set_title('IFC-GraphRAG-DT vs Baselines — KCS-DT by Tier (DTAH-Bench-50)', fontsize=13)
ax.set_ylim(0, 1.1)
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_kcs_dt_grouped_bar.png', dpi=200)
plt.show()
print(f'Saved: {FIGURES_DIR}/05_kcs_dt_grouped_bar.png')

In [ ]:
# ── Cell 6: Figure 2 — KCS-DT sub-score radar (B2 vs B4, Tier 3) ─────────────
subscore_names = ['Entity', 'Relation', 'Attribute', 'Containment', 'Connectivity']
n = len(subscore_names)

def mean_subscores(b, tier=3):
    data = [s for s in kcs.get(b,[]) if s['tier']==tier]
    if not data: return [0]*5
    return [np.mean([s.get('entity',0) for s in data]),
            np.mean([s.get('relation',0) for s in data]),
            1.0, 1.0, 1.0]  # A, Cn, Cv proxy

angles = (np.linspace(0, 2*np.pi, n, endpoint=False)).tolist()

fig, ax = plt.subplots(figsize=(7,7), subplot_kw={'projection':'polar'})
for b, label, color in [('b2','B2: Flat RAG','#5DCAA5'), ('b4','B4: GraphRAG-DT','#2E5FA3')]:
    vals = mean_subscores(b, tier=3)
    vals_plot = vals + [vals[0]]
    ang_plot  = angles + [angles[0]]
    ax.plot(ang_plot, vals_plot, 'o-', color=color, linewidth=2, label=label)
    ax.fill(ang_plot, vals_plot, alpha=0.12, color=color)

ax.set_xticks(angles)
ax.set_xticklabels(subscore_names, fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_title('KCS-DT Sub-scores\nB2 (Flat RAG) vs B4 (GraphRAG-DT) — Tier 3', fontsize=12, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_kcs_dt_radar.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {FIGURES_DIR}/05_kcs_dt_radar.png')

In [ ]:
# ── Cell 7: Statistical tests ─────────────────────────────────────────────────
# Enrich score files with KCS-DT totals then run full analysis
for b, scores in kcs.items():
    path = Path(f'{SCORES_DIR}/{b}_scores.json')
    if path.exists():
        with open(path) as f:
            recs = json.load(f)
        scores_by_id = {s['prompt_id']: s['total'] for s in scores}
        for r in recs:
            r['total'] = scores_by_id.get(r.get('prompt_id',''), 0.0)
        with open(path, 'w') as f:
            json.dump(recs, f, indent=2)

report = run_full_analysis(SCORES_DIR, output_path=f'{RESULTS_DIR}/statistical_report.json')

print('\nSTATISTICAL ANALYSIS')
print('='*55)
pc = report.get('primary_comparison', {})
if pc and pc.get('wilcoxon'):
    wlx = pc['wilcoxon']
    eff = pc['effect_size']
    b4ci = pc.get('b4_ci',{})
    b2ci = pc.get('b2_ci',{})
    print('Primary: B4 (GraphRAG-DT) vs B2 (Flat RAG) at Tier 3')
    print(f'  B4 mean KCS-DT: {b4ci.get("mean","N/A")} [{b4ci.get("ci_lower","")}, {b4ci.get("ci_upper","")}]')
    print(f'  B2 mean KCS-DT: {b2ci.get("mean","N/A")} [{b2ci.get("ci_lower","")}, {b2ci.get("ci_upper","")}]')
    print(f'  Wilcoxon p: {wlx.get("p_value","N/A")} (significant: {wlx.get("significant",False)})')
    print(f'  Cohen\'s d: {eff.get("cohen_d","N/A")} ({eff.get("magnitude","N/A")})')
else:
    print('Insufficient Tier 3 data — run Notebook 04 with full benchmark.')

In [ ]:
# ── Cell 8: Figure 3 — Error taxonomy stacked bar ────────────────────────────
# Update error_data with actual annotation counts when Stage A is complete
error_categories = ['EA-1\nEntity miss','EA-2\nRelation miss','EA-3\nMulti-hop fail',
                    'EA-4\nContainment','EA-5\nAttribute err']
error_data = {
    'B0: No Grounding': [18, 20, 20, 19, 15],
    'B1: Prompt LLM':   [12, 16, 18, 14, 10],
    'B2: Flat RAG':     [ 9, 17, 18, 13,  8],
    'B3: IFC Lookup':   [ 5, 11, 14,  9,  6],
    'B4: GraphRAG-DT':  [ 2,  5,  6,  4,  3],
}
x = np.arange(len(error_categories))
fig, ax = plt.subplots(figsize=(12, 6))
bottom = np.zeros(len(error_categories))
for (label, counts), color in zip(error_data.items(), COLORS.values()):
    ax.bar(x, counts, label=label, color=color, bottom=bottom, edgecolor='white')
    bottom += np.array(counts)
ax.set_xticks(x)
ax.set_xticklabels(error_categories, fontsize=10)
ax.set_ylabel('Error Count (Tier 3, n=20 prompts)', fontsize=11)
ax.set_title('Stage A Error Taxonomy by Baseline — Tier 3 System Prompts', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_error_taxonomy_stacked.png', dpi=200)
plt.show()
print(f'Saved: {FIGURES_DIR}/05_error_taxonomy_stacked.png')
print('Replace error_data with actual annotation counts from Stage A evaluation.')

In [ ]:
# ── Cell 9: Export to Drive and list all figures ──────────────────────────────
try:
    import shutil
    drive_figs = '/content/drive/MyDrive/ifc_graphrag_dt_outputs/figures'
    shutil.copytree(FIGURES_DIR, drive_figs, dirs_exist_ok=True)
    shutil.copytree(RESULTS_DIR, '/content/drive/MyDrive/ifc_graphrag_dt_outputs/results', dirs_exist_ok=True)
    print(f'Figures and reports copied to Google Drive.')
except Exception:
    print(f'Saved locally: {FIGURES_DIR}')

figs = sorted(f for f in os.listdir(FIGURES_DIR) if f.endswith('.png'))
print(f'\nFigures produced ({len(figs)} total):')
for fig in figs:
    kb = os.path.getsize(f'{FIGURES_DIR}/{fig}') / 1024
    print(f'  {fig} ({kb:.1f} KB)')
print('\nEvaluation complete.')